# Example Interlock Script
J. Ramirez 02-13-2026

**Brief: Subscribe to GeecsDevice and use ```InterlockServer``` class to create software interlock for your experiment.**

The InterlockServer class creates a tcp server to interface with Master Control. A user creates a server by setting the host IP and port to listen on in ```InterlockServer```.

For this example, we will create a simple camera threshold interlock by:
  1. Subscribing to an experiment device variable using the ```geecs_python_api```
  2. Creating a factory function to monitor the variable
  3. Call the interlock server class and setting a host IP/port to listen on
  4. Registering the factory function. and starting the server. 

We'll start by importing our necessary libraries/classes...

In [99]:
import time
from geecs_python_api.controls.interface.geecs_database import GeecsDatabase
from geecs_python_api.controls.devices.geecs_device import GeecsDevice
from geecs_python_api.controls.interlocks.geecs_interlock_server import InterlockServer
from dotenv import load_dotenv
import os

load_dotenv()

SERVER_IP = os.getenv("SERVER_IP")
SERVER_PORT = int(os.getenv("SERVER_PORT"))

Next, we need to access our experiment and subscribe to the camera variable. A list of experiment device variables is listed in the experiment editor. Each of these can be subscribed to via the subscribe_var_values() method and can be called/read via get().

In [100]:
# Set up the camera device and subscribe to the MaxCounts variable
GeecsDevice.exp_info = GeecsDatabase.collect_exp_info("BELLA")
cam1 = GeecsDevice("CAM-PL1-TapeDrivePointing")
cam1.use_alias_in_TCP_subscription = False
cam1.subscribe_var_values(
    [
        "MaxCounts",
        "MeanCounts",
        "acq_timestamp",
        "Target.X",
        "Target.Y",
        "centroidx",
        "centroidy",
    ]
)

print(cam1.state)  # verify connection
print(cam1.dev_tcp)
# check if camera analysis mode is on
if cam1.get("Analysis") != "on":
    print(
        "Camera analysis mode is not on. Please turn on camera analysis to use the interlock server."
    )

{'fresh': False, 'shot number': None, 'GEECS device error': False, 'Device alive on instantiation': True}


Once we have subscribed to the variables we want, we can define a function to set our interlock logic. In general, this can be any boolean condition, i.e., any combination of conditions that will ultimately cause the server to return "Safe" or "Unsafe." The server maps "True" to "warning" and "False" to "safe".

In this example, we want to know if the max counts on the camera. We define a helper function to collect the camera, the variable we want, and the threshold. It returns check() which toggles the boolean. 

In [101]:
def camera_thresh_check(
    camera, variable_name: str, thresh: float, timeout: float = 5000
):
    """Returns a function that checks if the specified camera variable is below a threshold, with timeout for stale data."""
    last_device_timestamp = None
    last_check_time = time.time()

    def check():
        nonlocal last_device_timestamp, last_check_time
        current_time = time.time()

        device_state = camera.state
        value = device_state[variable_name]
        device_timestamp = device_state["acq_timestamp"]

        if device_timestamp is not None:
            if device_timestamp == last_device_timestamp:
                time_frozen = current_time - last_check_time
                if time_frozen > timeout:
                    print(
                        f"WARNING: No new {variable_name} data received for {time_frozen:.1f} seconds. Device may be offline or unresponsive."
                    )
                    return True  # Return True (unsafe) if data is stale
            else:
                last_device_timestamp = device_timestamp
                last_check_time = current_time
        if value is None:
            print(
                f"WARNING: No {variable_name} data received. Device may be offline or unresponsive."
            )
            return True  # Return True (unsafe) if no value

        # if the value is below a threshold, raise interlock
        # print(f"{variable_name}: {value}")
        # print timestamp
        # print(f"Timestamp: {device_timestamp}")
        # if value < thresh:
        #     print(
        #         f"Camera {variable_name} value {value} is below threshold {thresh}. Interlock error."
        #     )
        # else:
        #     print(
        #         f"Camera {variable_name} value {value} is above threshold {thresh}. Interlock safe."
        #     )
        return value < thresh

    return check

In [102]:
def check_aligned(value, thresh, target):
    if abs(value - target) < thresh:
        return False  # aligned -> safe
    else:
        return True  # not aligned -> unsafe

In [103]:
from functools import partial

variable_names = ["MaxCounts", "MeanCounts", "centroidx", "centroidy"]
threshes = [800, 0, float(5), float(5)]
truth_functions = [
    lambda v, t: v < t,
    lambda v, t: v < t,
    partial(check_aligned, target=cam1.get("Target.X")),
    partial(check_aligned, target=cam1.get("Target.Y")),
]

In [104]:
def camera_thresh_check_multi(
    camera,
    variable_names: list[str],
    truth_functions: list,
    threshes: list[float],
    timeout: float = 5000,
):
    """Returns a function that returns True (unsafe) if ANY variable's predicate
    is True, or if data is stale/missing.
    """
    last_device_timestamp = None
    last_check_time = time.time()

    def check():
        nonlocal last_device_timestamp, last_check_time
        current_time = time.time()

        device_state = camera.state
        device_timestamp = device_state["acq_timestamp"]

        # Staleness check once per call (timestamp is shared across variables)
        if device_timestamp is not None:
            if device_timestamp == last_device_timestamp:
                time_frozen = current_time - last_check_time
                if time_frozen > timeout:
                    print(
                        f"WARNING: No new data received for {time_frozen:.1f} "
                        f"seconds. Device may be offline or unresponsive."
                    )
                    return True  # stale -> unsafe
            else:
                last_device_timestamp = device_timestamp
                last_check_time = current_time

        for variable_name, truth_function, thresh in zip(
            variable_names, truth_functions, threshes
        ):
            value = device_state[variable_name]
            if value is None:
                print(
                    f"WARNING: No {variable_name} data received. "
                    f"Device may be offline or unresponsive."
                )
                return True  # missing -> unsafe
            if truth_function(value, thresh):
                print(f"WARNING: {variable_name} value {value} failed its check. Interlock unsafe.")
                return True  # this variable is unsafe

        return False  # all good -> safe

    return check

Here, we create the server ```InterlockServer``` and give a host IP and port. We'll register a monitor via ```register_monitor()``` and pass ```camera_thresh_check()``` to it. Then we start the server with ```start()```. From there, the server will now broadcast _safe_ or _warning_ depending on our interlock logic.

In [105]:
server = InterlockServer(host=SERVER_IP, port=SERVER_PORT)  # create the interlock
# server.register_monitor("Camera MeanCounts Check", camera_thresh_check(cam1, "MeanCounts", 150),interval = 0.1)  # add the camera mean counts check to the interlock server with a threshold of 6000
# server.register_monitor("Camera MaxCounts Check", camera_thresh_check(cam1, "MaxCounts", 3000), interval=0.1)  # add the camera max counts check to the interlock server
server.register_monitor(
    "Camera Multi Check",
    camera_thresh_check_multi(cam1, variable_names, truth_functions, threshes),
    interval=0.1,
)
server.start()
print("Camera interlock server running...")
try:
    while True:
        # print(cam1.state["acq_timestamp"])
        time.sleep(0.02)
        # print(cam1.state["MaxCounts"]) # print the current value of MeanCounts to verify we can read it
except KeyboardInterrupt:
    server.stop()

Camera interlock server running...
